# Unlearning Extraction

Compare base vs unlearned model activations to see if unlearning has a directional bias.
- directional bias -> can try to reverse with ablation
- no delta -> refusal could be epistemic or nonlinear

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from datasets import load_dataset

from src.model_loader import load_model, get_device
from src.hooks import extract_activations_multi

Load Forget-Set Questions

In [2]:
forget_data = load_dataset("locuslab/TOFU", "forget10")["train"]
forget_questions = [x["question"] for x in forget_data]
print(f"Forget-set questions: {len(forget_questions)}")
print(f"Example: {forget_questions[0]}")

Forget-set questions: 400
Example: What is the full name of the author born in Taipei, Taiwan on 05/11/1991 who writes in the genre of leadership?


Load Models

In [ ]:
import torch, gc

ALL_LAYERS = [f"model.layers.{i}" for i in range(16)]

CHECKPOINTS = {
    "GradDiff": "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_GradDiff_lr1e-05_alpha5_epoch5",
    "NPO":      "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_NPO_lr1e-05_beta0.5_alpha1_epoch10",
    "RMU":      "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_RMU_lr5e-05_layer10_scoeff10_epoch10",
}
BASE_MODEL = "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"

def coherence(diffs):
    units = diffs / np.linalg.norm(diffs, axis=1, keepdims=True)
    cos = units @ units.T
    off = cos[~np.eye(len(diffs), dtype=bool)]
    return float(off.mean())

# Base activations at every layer — one model load, one pass per question
base_model, tokenizer, device = load_model(BASE_MODEL)

base_multi = extract_activations_multi(
    base_model, tokenizer, forget_questions, ALL_LAYERS, device
)
del base_model; gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Per-method, per-layer coherence (raw and mean-centered)
scan = {}
for name, path in CHECKPOINTS.items():
    m, tok, dev = load_model(path)
    unl_multi = extract_activations_multi(m, tok, forget_questions, ALL_LAYERS, dev)
    del m; gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()

    scan[name] = {}
    for layer in ALL_LAYERS:
        diffs = unl_multi[layer] - base_multi[layer]
        centered = diffs - diffs.mean(axis=0, keepdims=True)
        scan[name][layer] = {
            "coherence": coherence(diffs),
            "coherence_centered": coherence(centered),
        }
    print(f"done: {name}")

# Assemble a readable table
import pandas as pd
rows = []
for layer in ALL_LAYERS:
    idx = int(layer.split(".")[-1])
    row = {"layer": idx}
    for name in CHECKPOINTS:
        row[f"{name}"] = scan[name][layer]["coherence"]
        row[f"{name}_c"] = scan[name][layer]["coherence_centered"]
    rows.append(row)

df_scan = pd.DataFrame(rows)
pd.set_option("display.precision", 3)
print(df_scan.to_string(index=False))

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/tofu_Llama-3.2-1B-Instruct_full
Device: mps | dtype: torch.float16
Params: 1.2B


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_GradDiff_lr1e-05_alpha5_epoch5
Device: mps | dtype: torch.float16
Params: 1.2B
done: GradDiff


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_NPO_lr1e-05_beta0.5_alpha1_epoch10
Device: mps | dtype: torch.float16
Params: 1.2B
done: NPO


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_RMU_lr5e-05_layer10_scoeff10_epoch10
Device: mps | dtype: torch.float16
Params: 1.2B
done: RMU
 layer  GradDiff  GradDiff_c   NPO      NPO_c   RMU  RMU_c
     0     0.884   7.381e-03 0.876  7.190e-03   NaN    NaN
     1     0.753  -4.474e-04 0.727  6.459e-04   NaN    NaN
     2     0.673  -1.556e-03 0.676 -9.328e-04   NaN    NaN
     3     0.512  -2.263e-03 0.446 -1.508e-03   NaN    NaN
     4     0.387  -2.097e-03 0.301 -1.002e-03   NaN    NaN
     5     0.344  -2.208e-03 0.233 -7.312e-04   NaN    NaN
     6     0.287  -2.211e-03 0.206 -9.112e-04   NaN    NaN
     7     0.358  -2.141e-03 0.167 -7.197e-04   NaN    NaN
     8     0.378  -2.083e-03 0.142 -1.296e-03 0.948 -0.001
     9     0.372  -2.098e-03 0.139 -1.154e-03 0.624 -0.002
    10     0.356  -2.069e-03 0.132 -1.269e-03 0.823 -0.002
    11     0.338  -2.006e-03 0.117 -6.540e-04 0.713 -0.002
    12     0.326  -1.889e-03 0.110 -9.776e-04 0.632 -0.002
    13     

/var/folders/1l/rwxq81q945x_8d2xvj7r5s4w0000gn/T/ipykernel_45348/2259808091.py:12: RuntimeWarning: invalid value encountered in divide
  units = diffs / np.linalg.norm(diffs, axis=1, keepdims=True)


Directional Bias Test

Control

In [8]:
# A random baseline: what coherence would we expect by chance
# for random unit vectors in this dimensionality?
np.random.seed(42)
rand = np.random.randn(*diffs.shape)
rand_unit = rand / np.linalg.norm(rand, axis=1, keepdims=True)
rand_cos = (rand_unit @ rand_unit.T)[~np.eye(len(diffs), dtype=bool)]

print(f"Observed mean coherence: {mean_coherence:.3f}")
print(f"Random baseline:         {float(rand_cos.mean()):.3f} ± {float(rand_cos.std()):.3f}")

Observed mean coherence: 0.297
Random baseline:         0.000 ± 0.022
